### Inkrementalni vnos

- zazna NOVE vnose
- za NOVE vnose naredi novo različico
- ne zazna posodabljanja in brisanja vnosov

In [28]:
import dlt
from datetime import datetime, timezone
from uuid import uuid4

from mongodb import mongodb

In [29]:
SOURCE_SYSTEM = "mongodb_atlas"

DATABASE_NAME = "test"

BUCKET_NAME = "book-library"

INGESTION_TYPE = "incremental"

INGESTED_AT = datetime.now(timezone.utc)

RUN_ID = str(uuid4())

# za prvi incremental vnos
INITIAL_CURSOR = datetime.now(timezone.utc)

print("Load ID:", RUN_ID)
print("Začetni kazalec:", INITIAL_CURSOR)

Load ID: a5849d5b-bce8-483b-9315-f626e4d9f2e7
Začetni kazalec: 2026-11-03 07:15:18.312743+00:00


In [30]:
source = mongodb(
    database=DATABASE_NAME
).with_resources("authors", "books")

for resource_name, resource in source.resources.items():

    resource.apply_hints(
        incremental=dlt.sources.incremental(
            "updatedAt", # kazalni stolpec
            initial_value=INITIAL_CURSOR
        ),
        write_disposition="append", # novi podatki se DODAJO
        primary_key="_id" # zaznava ponovljenih zapisov
    )

    print(
        f"Zbirka {resource_name}: "
        "inkrementalni prenos po updatedAt"
    )

Zbirka authors: inkrementalni prenos po updatedAt
Zbirka books: inkrementalni prenos po updatedAt


In [31]:
def add_metadata(resource_name):

    def mapper(item):

        item = dict(item)

        item["_source_system"] = SOURCE_SYSTEM
        item["_source_resource"] = resource_name
        item["_ingestion_type"] = INGESTION_TYPE
        item["_ingested_at"] = INGESTED_AT
        item["_load_id"] = RUN_ID

        return item

    return mapper


for resource_name, resource in source.resources.items():

    resource.add_map(
        add_metadata(resource_name)
    )

print("Metapodatki so dodani.")

Metapodatki so dodani.


In [32]:
from dlt.destinations import filesystem

destination = filesystem(
    bucket_url=f"s3://{BUCKET_NAME}",

    layout=(
        "{table_name}/"
        "ingestion_date={YYYY}-{MM}-{DD}/"
        "{load_id}.{file_id}.{ext}"
    ),

    current_datetime=INGESTED_AT
)

In [33]:
pipeline = dlt.pipeline(
    pipeline_name="mongodb_to_silo",
    destination=destination,
    dataset_name=SOURCE_SYSTEM
)

print("Inkrementalni cevovod je pripravljen.")

Inkrementalni cevovod je pripravljen.


In [34]:
info = pipeline.run(
    source,
    write_disposition="append",
    loader_file_format="parquet"
)

print(info)

Pipeline mongodb_to_silo load step finished in 0.13 seconds
1 load package(s) were loaded to destination filesystem and into dataset mongodb_atlas
The filesystem destination used s3://book-library location to store data
Load package 1793690123.1012518 is LOADED and contains no failed jobs
